In [ ]:

# ─── CELL 1 — Imports ─────────────────────────────────────────────
import os, pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd

from config_and_utils import (
    clean_text, MaxoutLayer,
    DATASET_PATH, DATA_DIR, MODEL_DIR,
    JOINT_MODEL_PATH, EXTRACTOR_PATH, MAXOUT_PATH, TOKENIZER_PATH,
    VOCAB_SIZE, MAX_SEQUENCE_LENGTH, EMBEDDING_DIM, LSTM_UNITS,
    BATCH_SIZE, EPOCHS, PATIENCE, LR, L2, LABEL_SMOOTHING,
    THRESHOLD, RANDOM_STATE, TEXT_COLS
)


In [ ]:

# ─── CELL 2 — Build Model ─────────────────────────────────────────
def build_joint_model(vocab_size, embedding_dim, lstm_units, max_seq_len, l2=1e-4):
    reg = regularizers.l2(l2)
    inp = layers.Input(shape=(max_seq_len,), name="token_input")

    x = layers.Embedding(vocab_size, embedding_dim, name="embedding")(inp)
    x = layers.SpatialDropout1D(0.2, name="spatial_dropout")(x)

    x = layers.Bidirectional(
            layers.LSTM(lstm_units,
                        kernel_regularizer=reg, recurrent_regularizer=reg,
                        name="lstm"),
            name="bilstm_layer"
        )(x)

    x = layers.Dropout(0.4, name="dropout_1")(x)
    x = layers.BatchNormalization(name="bn_1")(x)

    x = MaxoutLayer(256, num_pieces=2, l2=l2, name="maxout_1")(x)
    x = layers.Dropout(0.4, name="dropout_2")(x)

    x = MaxoutLayer(128, num_pieces=2, l2=l2, name="maxout_2")(x)
    x = layers.Dropout(0.3, name="dropout_3")(x)

    x = MaxoutLayer(64,  num_pieces=2, l2=l2, name="maxout_3")(x)
    x = layers.Dropout(0.2, name="dropout_4")(x)

    out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg, name="output")(x)

    model = models.Model(inputs=inp, outputs=out, name="Joint_BiLSTM_Maxout")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy",
                 tf.keras.metrics.AUC(name="auc"),
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")]
    )
    return model



In [ ]:

# ─── CELL 3 — Data Loading & Preprocessing ────────────────────────
def load_and_preprocess():
    os.makedirs(DATA_DIR,  exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    required = ["X_train.npy","X_val.npy","X_test.npy",
                "y_train.npy","y_val.npy","y_test.npy","tokenizer.pkl"]
    arrays_ok = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required)

    if arrays_ok:
        X_check = np.load(os.path.join(DATA_DIR, "X_train.npy"))
        if X_check.shape[1] != MAX_SEQUENCE_LENGTH:
            print(f"[DATA] Seq length mismatch. Re-running preprocessing.")
            arrays_ok = False

    if arrays_ok:
        print("[DATA] Loading existing preprocessed arrays ...")
        X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
        X_val   = np.load(os.path.join(DATA_DIR, "X_val.npy"))
        X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
        y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
        y_val   = np.load(os.path.join(DATA_DIR, "y_val.npy"))
        y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))
        with open(TOKENIZER_PATH, "rb") as f:
            tokenizer = pickle.load(f)
        print(f"      Train:{X_train.shape}  Val:{X_val.shape}  Test:{X_test.shape}")
        return X_train, X_val, X_test, y_train, y_val, y_test, tokenizer

    print("[DATA] Running full preprocessing on EMSCAD dataset ...")
    df = pd.read_csv(DATASET_PATH)
    print(f"      Dataset shape : {df.shape}")

    for col in TEXT_COLS:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("")

    labels = df["fraudulent"].astype(int).values
    print(f"      Real (0): {(labels==0).sum()}  |  Fake (1): {(labels==1).sum()}")
    print(f"      Imbalance: {(labels==0).sum()/(labels==1).sum():.1f}:1")

    df["merged_text"] = (df["title"]           + " " +
                         df["company_profile"] + " " +
                         df["description"]     + " " +
                         df["requirements"]    + " " +
                         df["benefits"])

    print("      Cleaning text ...")
    df["cleaned_text"] = df["merged_text"].apply(clean_text)

    wc = df["cleaned_text"].apply(lambda x: len(x.split()))
    print(f"      Word count: mean={wc.mean():.0f}  median={wc.median():.0f}  "
          f"p90={wc.quantile(0.9):.0f}  max={wc.max()}")

    print(f"      Tokenizing (vocab={VOCAB_SIZE}) ...")
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(df["cleaned_text"].tolist())

    seqs   = tokenizer.texts_to_sequences(df["cleaned_text"].tolist())
    padded = pad_sequences(seqs, maxlen=MAX_SEQUENCE_LENGTH,
                           padding="post", truncating="post")
    print(f"      Padded shape: {padded.shape}")

    X_tv, X_test, y_tv, y_test = train_test_split(
        padded, labels, test_size=0.15, random_state=RANDOM_STATE, stratify=labels)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.15/0.85, random_state=RANDOM_STATE, stratify=y_tv)

    print(f"      Train:{X_train.shape}  Val:{X_val.shape}  Test:{X_test.shape}")

    for name, arr in [("X_train",X_train),("X_val",X_val),("X_test",X_test),
                      ("y_train",y_train),("y_val",y_val),("y_test",y_test)]:
        np.save(os.path.join(DATA_DIR, f"{name}.npy"), arr)
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(tokenizer, f)
    print("      All arrays and tokenizer saved.")

    return X_train, X_val, X_test, y_train, y_val, y_test, tokenizer



In [ ]:

# ─── CELL 4 — Train ───────────────────────────────────────────────
def train(X_train, X_val, y_train, y_val):
    print("\n[TRAIN] Building joint BiLSTM + Maxout model ...")
    model = build_joint_model(VOCAB_SIZE, EMBEDDING_DIM, LSTM_UNITS,
                              MAX_SEQUENCE_LENGTH, L2)
    model.summary()

    cw_arr       = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
    class_weight = {0: cw_arr[0], 1: cw_arr[1]}
    print(f"\n      Class weights: Real={class_weight[0]:.3f}  Fake={class_weight[1]:.3f}")

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE,
                      restore_best_weights=True, mode="min", verbose=1),
        ModelCheckpoint(JOINT_MODEL_PATH, monitor="val_loss",
                        save_best_only=True, mode="min", verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3,
                          min_lr=1e-6, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data = (X_val, y_val),
        epochs          = EPOCHS,
        batch_size      = BATCH_SIZE,
        class_weight    = class_weight,
        callbacks       = callbacks
    )
    return model, history


In [ ]:

# ─── CELL 5 — Evaluate ────────────────────────────────────────────
def evaluate(model, X_test, y_test, history):
    print("\n[EVAL] Test set evaluation ...")
    results      = model.evaluate(X_test, y_test, verbose=0)
    metric_names = [m.name for m in model.metrics]
    for name, val in zip(metric_names, results):
        print(f"      {name:15s}: {val:.4f}")

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob > THRESHOLD).astype(int)

    auc_score = roc_auc_score(y_test, y_prob)
    print(f"      {'roc_auc':15s}: {auc_score:.4f}")

    pred_dist = {("Real" if u==0 else "Fake"): int(c)
                 for u,c in zip(*np.unique(y_pred, return_counts=True))}
    print(f"\n      Prediction spread: {pred_dist}")

    print(f"\n  Classification Report (threshold={THRESHOLD}):")
    print(classification_report(y_test, y_pred,
                                target_names=["Real Job (0)", "Fake Job (1)"]))

    cm = confusion_matrix(y_test, y_pred)
    print("  Confusion Matrix:")
    print(f"                    Predicted Real   Predicted Fake")
    print(f"  Actual Real  :         {cm[0,0]:5d}            {cm[0,1]:5d}")
    print(f"  Actual Fake  :         {cm[1,0]:5d}            {cm[1,1]:5d}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (tr, va), title in zip(
        axes,
        [("accuracy","val_accuracy"),("loss","val_loss"),("auc","val_auc")],
        ["Accuracy","Loss","AUC"]
    ):
        ax.plot(history.history[tr], label="Train", linewidth=2)
        ax.plot(history.history[va], label="Val",   linewidth=2)
        ax.set_title(f"BiLSTM+Maxout — {title}", fontsize=12)
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("Training History — EMSCAD Real Dataset", fontsize=13, y=1.02)
    plt.tight_layout()
    plot_path = os.path.join(MODEL_DIR, "training_history.png")
    plt.savefig(plot_path, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"\n  Training plot saved -> {plot_path}")



In [ ]:

# ─── CELL 6 — Save Sub-Models ─────────────────────────────────────
def save_sub_models(joint_model):
    print("\n[SAVE] Extracting sub-models ...")

    bilstm_out = joint_model.get_layer("bilstm_layer").output
    extractor  = Model(inputs=joint_model.input, outputs=bilstm_out,
                       name="BiLSTM_Extractor")
    extractor.save(EXTRACTOR_PATH)
    print(f"      BiLSTM extractor  -> {EXTRACTOR_PATH}")

    bilstm_dim = extractor.output_shape[-1]
    feat_input = layers.Input(shape=(bilstm_dim,), name="features_input")
    x   = joint_model.get_layer("maxout_1")(feat_input)
    x   = joint_model.get_layer("dropout_2")(x)
    x   = joint_model.get_layer("maxout_2")(x)
    x   = joint_model.get_layer("dropout_3")(x)
    x   = joint_model.get_layer("maxout_3")(x)
    x   = joint_model.get_layer("dropout_4")(x)
    out = joint_model.get_layer("output")(x)
    maxout_model = Model(inputs=feat_input, outputs=out, name="Maxout_Classifier")
    maxout_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    maxout_model.save(MAXOUT_PATH)
    print(f"      Maxout classifier -> {MAXOUT_PATH}")



In [ ]:

# ─── CELL 7 — RUN EVERYTHING (execute this cell to train) ─────────
#     WARNING: Running this will TRAIN and OVERWRITE the saved model.
#     Only run this if you want to train from scratch.

X_train, X_val, X_test, y_train, y_val, y_test, _ = load_and_preprocess()
joint_model, history = train(X_train, X_val, y_train, y_val)
evaluate(joint_model, X_test, y_test, history)
joint_model.save(JOINT_MODEL_PATH)
print(f"\n      Joint model saved -> {JOINT_MODEL_PATH}")
save_sub_models(joint_model)
print("\n[DONE] Training complete. Now use predict_and_demo.ipynb for testing.")
